# NCDD OCR — KhmerOCR

Process one PDF with KhmerOCR, save TXT, and extract articles as JSON and CSV.

In [ ]:
%pip install git+https://github.com/seanghay/KhmerOCR

In [ ]:
import csv
import json
import re
import subprocess
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
TESTABLE_DIR = PROJECT_ROOT / 'data-pipeline' / 'testable'

import onnxruntime as ort
from tqdm.auto import tqdm

In [ ]:
INPUT_DIR = DATA_DIR / 'raw' / 'pdfs'
INPUT_PDF = INPUT_DIR / '17061.pdf'
OUTPUT_DIR = DATA_DIR / 'ocr' / 'test' / 'khmerocr'
TEXT_DIR = OUTPUT_DIR / 'text'
ARTICLE_JSON_DIR = OUTPUT_DIR / 'articles' / 'json'
ARTICLE_CSV_DIR = OUTPUT_DIR / 'articles' / 'csv'
for folder in (TEXT_DIR, ARTICLE_JSON_DIR, ARTICLE_CSV_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if not INPUT_PDF.exists():
    available = sorted(INPUT_DIR.glob('*.pdf'))
    if not available:
        raise FileNotFoundError(f'No PDF files found in {INPUT_DIR}')
    INPUT_PDF = available[0]
providers = ort.get_available_providers()
print('Test document:', INPUT_PDF)
print('ONNX Runtime providers:', providers)
if 'CoreMLExecutionProvider' not in providers:
    print('CoreML GPU provider is unavailable; KhmerOCR will use CPU.')

In [ ]:
ARTICLE_PATTERNS = (
    re.compile(r'^\s*(?:មាត្រា|មាត្រាទី)\s*([០-៩0-9]+)\s*[:.\-]?\s*(.*)$', re.IGNORECASE),
    re.compile(r'^\s*article\s*([0-9]+)\s*[:.\-]?\s*(.*)$', re.IGNORECASE),
)
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()
def extract_articles(text):
    articles, current = [], None
    for line in text.splitlines():
        line = clean_text(line)
        if not line: continue
        heading = next((p.match(line) for p in ARTICLE_PATTERNS if p.match(line)), None)
        if heading:
            if current:
                current['text'] = clean_text(' '.join(current.pop('_parts')))
                articles.append(current)
            current = {'article_number': heading.group(1), 'title': clean_text(heading.group(2)), 'text': '', '_parts': []}
        elif current:
            current['_parts'].append(line)
    if current:
        current['text'] = clean_text(' '.join(current.pop('_parts')))
        articles.append(current)
    return articles
def save_articles(articles, document_id):
    json_path = ARTICLE_JSON_DIR / f'{document_id}_articles.json'
    csv_path = ARTICLE_CSV_DIR / f'{document_id}_articles.csv'
    with json_path.open('w', encoding='utf-8') as file: json.dump(articles, file, ensure_ascii=False, indent=2)
    fields = ['article_number', 'title', 'text']
    with csv_path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields); writer.writeheader(); writer.writerows({f: a.get(f, '') for f in fields} for a in articles)
    return json_path, csv_path

In [ ]:
def process_one_document(pdf_path=INPUT_PDF, overwrite=False):
    pdf_path = Path(pdf_path); document_id = pdf_path.stem
    text_path = TEXT_DIR / f'{document_id}.txt'
    json_path = ARTICLE_JSON_DIR / f'{document_id}_articles.json'
    csv_path = ARTICLE_CSV_DIR / f'{document_id}_articles.csv'
    if not overwrite and all(path.exists() for path in (text_path, json_path, csv_path)):
        return {'document': pdf_path.name, 'status': 'skipped'}
    subprocess.run(['khmerocr', str(pdf_path), '--format', 'txt', '--output', str(text_path)], check=True)
    text = text_path.read_text(encoding='utf-8')
    articles = extract_articles(text); save_articles(articles, document_id)
    return {'document': pdf_path.name, 'status': 'success', 'articles': len(articles), 'text': str(text_path)}
result = process_one_document()
print(result)

In [ ]:
def process_all_documents(input_dir=INPUT_DIR, overwrite=False):
    results = []
    for pdf_path in tqdm(sorted(Path(input_dir).glob('*.pdf')), desc='KhmerOCR documents'):
        try: results.append(process_one_document(pdf_path, overwrite))
        except Exception as error: results.append({'document': pdf_path.name, 'status': 'failed', 'error': str(error)})
    return results
# all_results = process_all_documents()